In [1]:
import os
import pandas as pd
import numpy as np
import torch

from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from scipy.special import softmax

c:\Users\annmarle\Desktop\Redditi postituste automaatne klassifitseerimine erineva signaalitasemega keskkondades\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer = AutoTokenizer.from_pretrained(model_name)
config = AutoConfig.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Mudeli sildid: 0 -> Negative; 1 -> Neutral; 2 -> Positive
labels = config.id2label

# Siltide teisendamine numbriliseks skooriks, et hiljem arvutusi teha.
label_to_score = {
    "negative": -1,
    "neutral": 0, 
    "positive": 1
}

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1086.20it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def overall_sentiment(text):
    # Kui tekst puudub, jätame neutraalseks
    if not text:
        return {
            "label": "neutral",
            "numeric_score": 0.0,
            "negative": 0.0,
            "neutral": 1.0,
            "positive": 0.0,
        }

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        output = model(**encoded)

    sentiment_scores = output.logits[0].detach().numpy()
    sentiment_probabilities = softmax(sentiment_scores)

    label_probabilities = {}
    for i, prob in enumerate(sentiment_probabilities):
        label_probabilities[labels[i]] = prob

    
    predicted_index = np.argmax(sentiment_probabilities)
    predicted_label = labels[predicted_index]
    numeric_score = label_to_score[predicted_label]


    return {
        "label": predicted_label,
        "numeric_score": numeric_score,
        "negative": label_probabilities.get("negative", 0.0),
        "neutral": label_probabilities.get("neutral", 0.0),
        "positive": label_probabilities.get("positive", 0.0)
    }

In [ ]:
def analyze_row_for_overall(row):
    results = []

    post_score = row.score
    if pd.isna(post_score):
        post_score = 1
    post_score = max(1, int(post_score))

    title = row.title_clean 
    selftext = row.selftext_clean 
    post_text = f"{title} {selftext}".strip()

    post_pred = overall_sentiment(post_text)

    results.append({
        "post_id": row.id,
        "subreddit_name": row.subreddit_name,
        "source_type": "post",
        "target": "overall",
        "context_text": post_text,
        "score": post_score,
        "sentiment_label": post_pred["label"],
        "sentiment_score": post_pred["numeric_score"],
        "prob_negative": post_pred["negative"],
        "prob_neutral": post_pred["neutral"],
        "prob_positive": post_pred["positive"]
    })

    comments = row.comments

    for comment in comments:
        if not isinstance(comment, dict):
            continue

        comment_text = comment.get("body_clean", "")
        if not comment_text:
            continue

        comment_score = comment.get("score", 1)
        if pd.isna(comment_score):
            comment_score = 1
        comment_score = max(1, int(comment_score))

        if post_text and comment_text:
            comment_context = f"Post: {post_text} Comment: {comment_text}".strip()
        elif comment_text:
            comment_context = comment_text
        else:
            comment_context = post_text

        comment_pred = overall_sentiment(comment_context)

        results.append({
            "post_id": row.id,
            "subreddit_name": row.subreddit_name,
            "source_type": "comment",
            "target": "overall",
            "context_text": comment_context,
            "score": comment_score,
            "sentiment_label": comment_pred["label"],
            "sentiment_score": comment_pred["numeric_score"],
            "prob_negative": comment_pred["negative"],
            "prob_neutral": comment_pred["neutral"],
            "prob_positive": comment_pred["positive"]
        })

    return results

In [5]:
# Kaalutud keskmise leidmine, et populaarsemad postitused/kommentaarid mõjutaksid rohkem.
# sisend: postituse meelestatuse numbrilised väärtused ja nende upvote skoorid
# väljund: kaalutud keskmine
def weighted_mean(values, scores):
    return np.average(values, weights=scores)

def subreddit_summary(sentiment_df):
    rows = []
    for (subreddit, target), g in sentiment_df.groupby(["subreddit_name", "target"]):
        score = g["score"]

        row = {
            "subreddit_name": subreddit,
            "target": target,
            "weighted_avg_sentiment": weighted_mean(g["sentiment_score"], score),
            "n_texts": len(g),
            "weighted_mentions": score.sum(),
            "share_negative": weighted_mean(g["prob_negative"], score),
            "share_neutral": weighted_mean(g["prob_neutral"], score),
            "share_positive": weighted_mean(g["prob_positive"], score),
        }

        rows.append(row)

    summary_df = pd.DataFrame(rows)

    share_cols = [
        "share_negative",
        "share_neutral",
        "share_positive"
    ]

    # Leiame millist meelestatust oli kõige rohkem subredditis.. 
    summary_df["main_sentiment"] = summary_df[share_cols].idxmax(axis=1)
    summary_df["main_sentiment"] = summary_df["main_sentiment"].str.replace("share_", "")

    summary_df["subreddit_sentiment"] = summary_df["weighted_avg_sentiment"].apply(lambda x: "positive" if x > 0 else ("negative" if x < 0 else "neutral"))

    return summary_df


In [8]:

import random

data_low = pd.read_json("../data/labeled/labeled_clean_low.json")
data_high = pd.read_json("../data/labeled/labeled_clean_high.json")

random_states = []
all_summaries_overall = []
first_rs_details_rows = []

for i in range(5):
    print("Iteration:", i+1)
    random_state = random.randint(0, 100)

    while True:
        # Kontrollime, et sama juhuvalim ei korduks.
        if random_state in random_states:
            random_state = random.randint(0, 100)
            continue
        else:            

            random_states.append(random_state)

            data_high_sample = data_high.sample(n=len(data_low), random_state=random_state)
            data = pd.concat([data_low, data_high_sample], ignore_index=True)

            all_rows = []
            for row in data.itertuples():
                all_rows.extend(analyze_row_for_overall(row))

            sentiment_overall_data = pd.DataFrame(all_rows)
            summary_overall_data = subreddit_summary(sentiment_overall_data)
            summary_overall_data["random_state"] = random_state

            sentiment_overall_data["random_state"] = random_state
            sentiment_overall_data["method"] = "overall"
            if i == 0:
                first_rs_details_rows = sentiment_overall_data.copy()
            all_summaries_overall.append(summary_overall_data)
            break
            

all_summaries_overall_data = pd.concat(all_summaries_overall, ignore_index=True)
all_summaries_overall_data = all_summaries_overall_data.groupby(["subreddit_name"], as_index=False).agg({
    "weighted_avg_sentiment": "mean",
    "n_texts": "mean",
    "weighted_mentions": "mean",
    "share_negative": "mean",
    "share_neutral": "mean",
    "share_positive": "mean",
})

all_summaries_overall_data["method"] = "overall"



Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 5


In [9]:
import json

with open("random_states.json", "w") as f:
    json.dump(random_states, f)

In [10]:
os.makedirs("../results", exist_ok=True)
# save data to files
all_summaries_overall_data.to_csv("../results/overall_sentiment_summary.csv", index=False)
first_rs_details_rows.to_csv("../results/overall_sentiment_first_random_state_details.csv", index=False)


In [12]:
all_summaries_overall_data

,subreddit_name,weighted_avg_sentiment,n_texts,weighted_mentions,share_negative,share_neutral,share_positive,method
0,CRM,-0.010769,980.0,2600.0,0.204092,0.582481,0.213427,overall
1,DigitalMarketing,-0.010296,359.0,777.0,0.197027,0.546928,0.256045,overall
2,Entrepreneur,-0.950299,160.0,4185.0,0.643312,0.328184,0.028504,overall
3,b2bmarketing,0.110912,232.0,559.0,0.157486,0.577338,0.265176,overall
4,hubspot,-0.096126,1959.8,4555.6,0.254816,0.561948,0.183236,overall
5,marketing,-0.180457,90.0,1269.0,0.412219,0.496710,0.091071,overall
6,sales,0.193652,149.0,3434.0,0.232799,0.293923,0.473278,overall
7,startups,-0.206349,39.0,126.0,0.377225,0.552600,0.070175,overall
8,techsales,-0.012435,134.0,965.0,0.149184,0.677034,0.173782,overall


In [15]:
first_rs_details_rows.tail()

,post_id,subreddit_name,source_type,target,context_text,score,sentiment_label,sentiment_score,prob_negative,prob_neutral,prob_positive,random_state,method
4075,1qv4vsy,hubspot,comment,overall,"Post: HubSpot Support Agent Hi all,\n\nI run a...",2,neutral,0,0.066061,0.727861,0.206079,97,overall
4076,1qv4vsy,hubspot,comment,overall,"Post: HubSpot Support Agent Hi all,\n\nI run a...",1,neutral,0,0.068460,0.743675,0.187865,97,overall
4077,1qv4vsy,hubspot,comment,overall,"Post: HubSpot Support Agent Hi all,\n\nI run a...",1,neutral,0,0.066260,0.742701,0.191038,97,overall
4078,1qv4vsy,hubspot,comment,overall,"Post: HubSpot Support Agent Hi all,\n\nI run a...",1,neutral,0,0.066541,0.747964,0.185495,97,overall
4079,1qv4vsy,hubspot,comment,overall,"Post: HubSpot Support Agent Hi all,\n\nI run a...",1,neutral,0,0.102287,0.788577,0.109135,97,overall
